# Task 4 — Health Risk Baseline (v1_1, post-review fixes)

Fixes a real bug from v1_0: synthetic health labels were generated per HOUR and mislabeled as `daily_mortality`, inflating the apparent sample size (194/48 hourly rows) when the real sample is 10 labeled ward-days. Runs at correct daily granularity now.

**Synthetic data disclaimer**: labels are synthetically generated (see `docs/science/health_risk_model.md`).

In [1]:
import sys; sys.path.insert(0, '..')
import pandas as pd
from src.health.feature_builder import build_features
from src.health.daily_aggregation import aggregate_to_daily
from src.health.generate_synthetic_health import generate_synthetic_health
from src.health.baseline_risk import drop_unlabeled, time_based_split, train_baseline_models, evaluate
from src.health.train_models import run_training
from src.health.predict import predict_health_risk

## 1. Build features and aggregate to daily (one row per ward-day)

In [2]:
features = build_features('../data/processed/thermal/stage1_output.csv', '../data/processed/vulnerability/ward_vulnerability.csv')
daily = aggregate_to_daily(features)
print(f'{len(daily)} ward-day rows (was 242 hourly rows pre-fix)')
daily[['ward_id', 'date', 'thermal_stress_mean', 'vulnerability_0_100', 'quality_flag']]

12 ward-day rows (was 242 hourly rows pre-fix)


,ward_id,date,thermal_stress_mean,vulnerability_0_100,quality_flag
0,GHMC_W001,2026-05-10,68.423750,61.59,RADIATION_APPROXIMATED
1,GHMC_W001,2026-05-11,69.972083,61.59,RADIATION_APPROXIMATED
2,GHMC_W001,2026-05-12,69.927083,61.59,RADIATION_APPROXIMATED
3,GHMC_W001,2026-05-13,68.287083,61.59,RADIATION_APPROXIMATED
4,GHMC_W001,2026-05-14,66.031250,61.59,RADIATION_APPROXIMATED
5,GHMC_W002,2026-05-10,72.745000,85.00,RADIATION_APPROXIMATED
6,GHMC_W002,2026-05-11,74.238333,85.00,RADIATION_APPROXIMATED
7,GHMC_W002,2026-05-12,74.148750,85.00,RADIATION_APPROXIMATED
8,GHMC_W002,2026-05-13,72.543333,85.00,RADIATION_APPROXIMATED
9,GHMC_W002,2026-05-14,70.282500,85.00,RADIATION_APPROXIMATED


## 2. Generate synthetic daily health labels (one label per ward-day)

In [3]:
synthetic = generate_synthetic_health(daily)
labeled = drop_unlabeled(synthetic)
print(f'{len(labeled)} ward-days with a valid label (missing-vulnerability rows correctly excluded)')
labeled[['ward_id', 'date', 'daily_mortality', 'daily_hospitalizations']]

10 ward-days with a valid label (missing-vulnerability rows correctly excluded)


,ward_id,date,daily_mortality,daily_hospitalizations
0,GHMC_W001,2026-05-10,0.0,1.0
1,GHMC_W001,2026-05-11,0.0,1.0
2,GHMC_W001,2026-05-12,0.0,2.0
3,GHMC_W001,2026-05-13,0.0,0.0
4,GHMC_W001,2026-05-14,0.0,3.0
5,GHMC_W002,2026-05-10,1.0,1.0
6,GHMC_W002,2026-05-11,0.0,0.0
7,GHMC_W002,2026-05-12,0.0,3.0
8,GHMC_W002,2026-05-13,0.0,0.0
9,GHMC_W002,2026-05-14,0.0,1.0


## 3. Train/validation/test split and baseline fit
**Honest warning**: ~10 labeled ward-days gives single-digit split sizes. These metrics are a pipeline sanity check, not a skill estimate.

In [4]:
train, val, test = time_based_split(labeled)
print(f'train={len(train)}, val={len(val)}, test={len(test)}')
models = train_baseline_models(train)
print('Validation:'); display(evaluate(models, val))
print('Test:'); display(evaluate(models, test))

train=6, val=2, test=2
Validation:


,target,mean_poisson_deviance,mae,n
0,daily_mortality,1.887201,0.943609,2
1,daily_hospitalizations,2.791780,1.395905,2


Test:


,target,mean_poisson_deviance,mae,n
0,daily_mortality,2977.626165,1488.813099,2
1,daily_hospitalizations,0.973107,1.057158,2


## 4. Save trained models, then run the served pipeline (what Task 5 calls)

In [5]:
meta = run_training(
    '../data/processed/thermal/stage1_output.csv',
    '../data/processed/vulnerability/ward_vulnerability.csv',
    models_dir='../models',
)
print(meta['warning'])

Sample size is far too small for a real skill estimate — pipeline sanity check only.


In [6]:
records = predict_health_risk(
    '../data/processed/thermal/stage1_output.csv',
    '../data/processed/vulnerability/ward_vulnerability.csv',
    models_dir='../models',
)
records

,ward_id,valid_time_utc,forecast_day,thermal_stress_mean,vulnerability_0_100,mortality_risk_0_100,hospitalization_risk_0_100,mortality_risk_level,hospitalization_risk_level,mortality_drivers,hospitalization_drivers,is_synthetic,source_type,model_version,quality_flag
0,GHMC_W001,2026-05-10 23:00:00+00:00,1,68.423750,61.59,4.076700e-07,48.207057,low,moderate,"thermal_stress_mean,thermal_stress_max,vulnera...","thermal_stress_max,thermal_stress_mean,doy_sin",True,synthetic_demo,baseline_poisson_v1_1,RADIATION_APPROXIMATED|FORECAST_DAY_APPROXIMATED
1,GHMC_W001,2026-05-11 23:00:00+00:00,2,69.972083,61.59,3.268230e-09,60.889022,low,high,"thermal_stress_mean,thermal_stress_max,vulnera...","thermal_stress_max,thermal_stress_mean,doy_sin",True,synthetic_demo,baseline_poisson_v1_1,RADIATION_APPROXIMATED|FORECAST_DAY_APPROXIMATED
2,GHMC_W001,2026-05-12 23:00:00+00:00,3,69.927083,61.59,3.779277e-09,64.402642,low,high,"thermal_stress_mean,thermal_stress_max,vulnera...","thermal_stress_max,thermal_stress_mean,doy_cos",True,synthetic_demo,baseline_poisson_v1_1,RADIATION_APPROXIMATED|FORECAST_DAY_APPROXIMATED
3,GHMC_W001,2026-05-13 23:00:00+00:00,4,68.287083,61.59,6.338029e-07,58.363159,low,high,"thermal_stress_mean,thermal_stress_max,vulnera...","thermal_stress_max,thermal_stress_mean,doy_cos",True,synthetic_demo,baseline_poisson_v1_1,RADIATION_APPROXIMATED|FORECAST_DAY_APPROXIMATED
4,GHMC_W001,2026-05-14 23:00:00+00:00,5,66.031250,61.59,9.838441e-04,54.965431,low,high,"thermal_stress_mean,thermal_stress_max,vulnera...","thermal_stress_max,thermal_stress_mean,doy_cos",True,synthetic_demo,baseline_poisson_v1_1,RADIATION_APPROXIMATED|FORECAST_DAY_APPROXIMATED
5,GHMC_W002,2026-05-10 23:00:00+00:00,1,72.745000,85.00,3.274630e-01,51.529857,low,high,"thermal_stress_mean,thermal_stress_max,vulnera...","thermal_stress_max,thermal_stress_mean,doy_sin",True,synthetic_demo,baseline_poisson_v1_1,RADIATION_APPROXIMATED|FORECAST_DAY_APPROXIMATED
6,GHMC_W002,2026-05-11 23:00:00+00:00,2,74.238333,85.00,2.526598e-03,59.707602,low,high,"thermal_stress_mean,thermal_stress_max,vulnera...","thermal_stress_max,thermal_stress_mean,doy_sin",True,synthetic_demo,baseline_poisson_v1_1,RADIATION_APPROXIMATED|FORECAST_DAY_APPROXIMATED
7,GHMC_W002,2026-05-12 23:00:00+00:00,3,74.148750,85.00,3.378364e-03,63.077334,low,high,"thermal_stress_mean,thermal_stress_max,vulnera...","thermal_stress_max,thermal_stress_mean,doy_cos",True,synthetic_demo,baseline_poisson_v1_1,RADIATION_APPROXIMATED|FORECAST_DAY_APPROXIMATED
8,GHMC_W002,2026-05-13 23:00:00+00:00,4,72.543333,85.00,6.282229e-01,61.677594,low,high,"thermal_stress_mean,thermal_stress_max,vulnera...","thermal_stress_max,thermal_stress_mean,doy_cos",True,synthetic_demo,baseline_poisson_v1_1,RADIATION_APPROXIMATED|FORECAST_DAY_APPROXIMATED
9,GHMC_W002,2026-05-14 23:00:00+00:00,5,70.282500,85.00,9.999520e+01,58.225751,severe,high,"thermal_stress_mean,thermal_stress_max,vulnera...","thermal_stress_max,thermal_stress_mean,doy_cos",True,synthetic_demo,baseline_poisson_v1_1,RADIATION_APPROXIMATED|FORECAST_DAY_APPROXIMATED


## Still-open limitations (see docs/science/health_risk_model.md)
- Only 10 labeled ward-days total — not enough for a real skill estimate.
- `forecast_day` is a historical-day stand-in, not a real forecast horizon.
- Ward mapping (`HYD_W001→GHMC_W001`) is an unconfirmed guess.
- Risk score is a project-defined, model-relative scale — not a clinical probability.